# 1. Konfigurasi (Configuration)
Bagian ini menyimpan konfigurasi utama untuk proses migrasi. Mode `DRY_RUN` diaktifkan secara default untuk menjamin keamanan. Tidak ada unggahan ke Firebase yang akan terjadi selama `DRY_RUN = True`.

In [1]:
# KONFIGURASI UTAMA
DRY_RUN = False
BATCH_SIZE = 100

# Jalur log lokal
LOG_DIRECTORY = "d:/Github/Projek_Rainfall/Analisis_Meteorologi/logs"

# URL Firebase RTDB
FIREBASE_URL = "https://staklimjerukagung-default-rtdb.asia-southeast1.firebasedatabase.app/"

print(f"DRY_RUN Mode: {DRY_RUN}")
print(f"Batch Size: {BATCH_SIZE}")

DRY_RUN Mode: False
Batch Size: 100


# 2. Pemeriksaan Dependensi (Dependency Check)
Pastikan Anda mengeksekusi ini di environment conda target: `(D:\conda_env\tensorflow)` dengan menjalankan `conda activate tensorflow` di terminal PowerShell Anda. Jika dependensi belum ada, jalankan sel di bawah ini (hilangkan komentar pada baris pip install).

In [2]:
# Menyiapkan library yang dibutuhkan
import sys
import subprocess

def install_dependencies():
    packages = ["firebase-admin", "pandas", "numpy", "tqdm"]
    print("Menginstal dependensi...")
    # subprocess.check_call([sys.executable, '-m', 'pip', 'install', *packages])
    print("Silakan hilangkan komentar pada perintah di atas jika belum menginstal dependensi.")

# install_dependencies()

import os
from pathlib import Path
import pandas as pd
import numpy as np
from tqdm.notebook import tqdm
import json
print("Dependensi siap!")

Dependensi siap!


# 3. Autentikasi Firebase (Firebase Authentication)
Sel ini memuat kredensial Firebase Anda. Kami menggunakan path spesifik dari sistem Anda.

In [3]:
import firebase_admin
from firebase_admin import credentials, db

# Jangan lakukan autentikasi ganda
if not firebase_admin._apps:
    try:
        cred = credentials.Certificate('D:/staklimjerukagung-firebase-adminsdk-kcfma-e091165a9b.json')
        firebase_admin.initialize_app(cred, {
            'databaseURL': FIREBASE_URL
        })
        print("Firebase berhasil diinisialisasi!")
    except FileNotFoundError:
        print("PERINGATAN: File JSON kredensial tidak ditemukan pada path D:/staklimjerukagung-firebase-adminsdk-kcfma-e091165a9b.json. Silakan cek ulang.")
else:
    print("Firebase sudah terinisialisasi.")

Firebase berhasil diinisialisasi!


# 4. Pencarian File (File Discovery)
Memindai direktori `/log` secara rekursif untuk mencari file berekstensi `.csv`, `.txt`, dan `.log` dan mengurutkannya secara kronologis.

In [4]:
def discover_files(log_dir):
    path = Path(log_dir)
    # Temukan semua file log
    extensions = ['*.csv', '*.txt', '*.log']
    file_list = []
    
    for ext in extensions:
        file_list.extend(path.rglob(ext))
        
    # Urutkan berdasarkan waktu modifikasi file
    file_list.sort(key=lambda x: x.stat().st_mtime)
    
    inventory = []
    for f in file_list:
        size_kb = f.stat().st_size / 1024
        inventory.append({
            "File": f.name,
            "Path": str(f),
            "Size (KB)": round(size_kb, 2)
        })
        
    df_inventory = pd.DataFrame(inventory)
    return file_list, df_inventory

files, df_inventory = discover_files(LOG_DIRECTORY)
print(f"Ditemukan {len(files)} file log.")
display(df_inventory.head())

Ditemukan 3 file log.


,File,Path,Size (KB)
0,2026-06-02.csv,d:\Github\Projek_Rainfall\Analisis_Meteorologi...,87.91
1,2026-06-04.csv,d:\Github\Projek_Rainfall\Analisis_Meteorologi...,57.59
2,2026-06-06.csv,d:\Github\Projek_Rainfall\Analisis_Meteorologi...,70.43


# 5. Parsing Data (Data Parsing)
Membaca file log dan memetakannya persis dengan skema `main.cpp`.
Skema Target Firebase:
```json
{
  "dew": 22.32,
  "humidity": 94.8,
  "pressure": 1009.36,
  "temperature": 23.2,
  "timestamp": 1780099969,
  "volt": 3.91,
  "rainfall": 0.0,
  "rainrate": 0.0,
  "tips": 0
}
```

In [6]:
# List penampung baris bermasalah
parsing_errors = []

def parse_record(row):
    try:
        # Menghapus 'ID-' atau 'id-' dan membuang leading zeros (Misal: ID-01 -> 1)
        raw_node = str(row['NodeID']).replace('ID-', '').replace('id-', '').strip()
        station_id = int(float(raw_node))
        
        # Ekstrak field dan atasi missing value dengan null (None)
        payload = {
            "dew": float(row['DewPoint']) if pd.notna(row['DewPoint']) else None,
            "humidity": float(row['Humidity']) if pd.notna(row['Humidity']) else None,
            "pressure": float(row['Pressure']) if pd.notna(row['Pressure']) else None,
            "temperature": float(row['Temperature']) if pd.notna(row['Temperature']) else None,
            "timestamp": int(row['Timestamp']),
            "volt": float(row['Voltage']) if pd.notna(row['Voltage']) else None,
            "rainfall": 0.0,
            "rainrate": 0.0,
            "tips": 0
        }
        
        # Tambahkan Lux dan DsTemperature HANYA untuk ID-01
        if station_id == 1:
            payload["lux"] = float(row['Lux']) if pd.notna(row['Lux']) else None
            payload["soil_temp"] = float(row['DsTemperature']) if pd.notna(row['DsTemperature']) else None
            
        return station_id, payload
    except Exception as e:
        parsing_errors.append({"row_data": row.to_dict(), "error": str(e)})
        return None, None

def parse_file(file_path):
    try:
        df = pd.read_csv(file_path, on_bad_lines='skip')
        
        # Validasi ketersediaan kolom kunci
        if 'Timestamp' not in df.columns or 'NodeID' not in df.columns:
            raise ValueError(f"Kolom wajib tidak ditemukan di {file_path}")
            
        parsed_records = []
        for _, row in df.iterrows():
            st_id, payload = parse_record(row)
            if st_id is not None:
                parsed_records.append((st_id, payload))
                
        return parsed_records
    except Exception as e:
        parsing_errors.append({"file": file_path, "error": str(e)})
        return []

all_records = []
for f in tqdm(files, desc="Parsing files"):
    all_records.extend(parse_file(str(f)))
    
print(f"Berhasil parsing {len(all_records)} baris.")
print(f"Jumlah baris error: {len(parsing_errors)}")

Parsing files:   0%|          | 0/3 [00:00<?, ?it/s]

Berhasil parsing 3230 baris.
Jumlah baris error: 0


# 6. Validasi Timestamp (Timestamp Validation)
Setiap record harus memiliki timestamp di atas `1700000000` (UTC unix epoch). Data yang tidak lolos akan disimpan di `invalid_records.csv`.

In [7]:
valid_records = []
invalid_timestamps = []

for st_id, payload in all_records:
    ts = payload.get("timestamp", 0)
    if ts > 1700000000:
        valid_records.append((st_id, payload))
    else:
        invalid_timestamps.append(payload)

print(f"Data valid: {len(valid_records)}")
print(f"Data invalid (dibuang): {len(invalid_timestamps)}")

if len(invalid_timestamps) > 0:
    df_inv = pd.DataFrame(invalid_timestamps)
    df_inv.to_csv("invalid_records.csv", index=False)
    print("Data invalid telah disimpan ke invalid_records.csv")

Data valid: 3230
Data invalid (dibuang): 0


# 7 & 8. Audit Kronologi & Pemetaan Stasiun (Timestamp Audit & Station Mapping)
Memeriksa apakah terdapat duplikasi waktu atau interval yang terbalik untuk masing-masing `station_id`.

In [8]:
from collections import defaultdict

# Memetakan data per stasiun (Station Mapping)
station_data = defaultdict(list)
for st_id, payload in valid_records:
    station_data[st_id].append(payload)

audit_report = []

for st_id in station_data.keys():
    # Sort secara kronologis
    station_data[st_id].sort(key=lambda x: x['timestamp'])
    
    timestamps = [x['timestamp'] for x in station_data[st_id]]
    
    # Deteksi duplikat
    duplicates = len(timestamps) - len(set(timestamps))
    
    # Deteksi waktu terbalik (Seharusnya 0 karena sudah kita urutkan)
    # Namun kita catat interval yang janggal
    diffs = np.diff(timestamps)
    missing_intervals = sum(1 for d in diffs if d > 3600) # Gap lebih dari 1 jam
    
    audit_report.append({
        "Station ID": st_id,
        "Total Records": len(timestamps),
        "Duplicates": duplicates,
        "Missing Intervals (>1h)": missing_intervals,
        "Start": timestamps[0],
        "End": timestamps[-1]
    })
    
    # Filter duplikat menggunakan dict
    unique_data = {item['timestamp']: item for item in station_data[st_id]}
    station_data[st_id] = list(unique_data.values())

df_audit = pd.DataFrame(audit_report)
display(df_audit)

,Station ID,Total Records,Duplicates,Missing Intervals (>1h),Start,End
0,1,1593,0,3,1780364484,1780773626
1,3,1637,0,3,1780364524,1780773662


# 9. Laporan Simulasi (Dry Run Report)
Mengumpulkan hasil audit menjadi file `migration_summary.md` yang harus direview sebelum proses dilanjutkan.

In [9]:
def generate_summary():
    total_valid = sum(len(v) for v in station_data.values())
    
    report = f"""# MIGRATION SUMMARY REPORT
    
## 1. File Statistics
- **Total Files Scanned:** {len(files)}
- **Total Rows Parsed:** {len(all_records)}
- **Valid Rows for Upload:** {total_valid}
- **Invalid Rows (Rejected):** {len(invalid_timestamps)}
- **Parsing Errors:** {len(parsing_errors)}

## 2. Station Counts
"""
    for st_id, records in station_data.items():
        report += f"- **Station {st_id}:** {len(records)} records\n"
        
    report += f"""
## 3. Upload Estimates
- **Batches needed (Size {BATCH_SIZE}):** {total_valid // BATCH_SIZE + 1}
- **Estimated API Calls:** {total_valid // BATCH_SIZE + 1}

*Please execute the upload cell only after reviewing this summary.*
"""
    
    with open("migration_summary.md", "w") as f:
        f.write(report)
        
    print(report)

generate_summary()

# MIGRATION SUMMARY REPORT

## 1. File Statistics
- **Total Files Scanned:** 3
- **Total Rows Parsed:** 3230
- **Valid Rows for Upload:** 3230
- **Invalid Rows (Rejected):** 0
- **Parsing Errors:** 0

## 2. Station Counts
- **Station 1:** 1593 records
- **Station 3:** 1637 records

## 3. Upload Estimates
- **Batches needed (Size 100):** 33
- **Estimated API Calls:** 33

*Please execute the upload cell only after reviewing this summary.*



# 10. Eksekusi Unggahan (Upload Execution)
**PERHATIAN**: Sel ini HANYA mengeksekusi unggahan jika `DRY_RUN = False`.
Data diunggah dalam ukuran batch tertentu untuk mencegah beban berlebih.
Format Path Firebase: `/auto_weather_stat/id-0{station_id}/data/{timestamp}`

In [10]:
def batch_upload():
    if DRY_RUN:
        print("MENGHENTIKAN UNGGAHAN: DRY_RUN masih aktif (True). Ubah ke False untuk mengunggah ke database.")
        return
        
    print("Memulai Migrasi ke Firebase...")
    
    for st_id, records in station_data.items():
        print(f"Mengunggah Station {st_id}...")
        
        # Batching
        for i in tqdm(range(0, len(records), BATCH_SIZE)):
            batch = records[i:i + BATCH_SIZE]
            
            # Kita menggunakan update_dict untuk mengurangi jumlah koneksi
            # Dictionary berisi child path sebagai key
            update_dict = {}
            for rec in batch:
                ts = rec['timestamp']
                # Persis dengan logika main.cpp format string id-0%d
                path = f"/auto_weather_stat/id-0{st_id}/data/{ts}"
                update_dict[path] = rec
                
            # Mendorong (Push) multiple records sekaligus
            db.reference('/').update(update_dict)
            
    print("Migrasi selesai!")

# JANGAN UBAH DRY_RUN SEBELUM MEMERIKSA SUMMARY REPORT
# DRY_RUN = False 
batch_upload()

Memulai Migrasi ke Firebase...
Mengunggah Station 1...


  0%|          | 0/16 [00:00<?, ?it/s]

Mengunggah Station 3...


  0%|          | 0/17 [00:00<?, ?it/s]

Migrasi selesai!


# 11. Verifikasi Pasca Unggah (Post Upload Verification)
Mengecek beberapa node sampel di Firebase untuk memverifikasi struktur schema.

In [11]:
def verify_upload():
    if DRY_RUN:
        print("Dry run aktif, tidak ada data di Firebase yang perlu diverifikasi.")
        return
        
    for st_id in station_data.keys():
        print(f"\nMengecek sample data untuk Station {st_id}...")
        # Tambahkan .order_by_key() sebelum .limit_to_last(1)
        ref = db.reference(f'/auto_weather_stat/id-0{st_id}/data').order_by_key().limit_to_last(1)
        snapshot = ref.get()
        print(json.dumps(snapshot, indent=2))

verify_upload()


Mengecek sample data untuk Station 1...
{
  "1780807366": {
    "dew": 26.88843,
    "humidity": 71.01,
    "lux": 5801.66,
    "pressure": 1012.72,
    "rainfall": 0,
    "rainrate": 0,
    "soil_temp": 26.43,
    "temperature": 32.84,
    "timestamp": 1780807366,
    "volt": 3.88
  }
}

Mengecek sample data untuk Station 3...
{
  "1780807375": {
    "dew": 24.51454,
    "humidity": 63.34,
    "pressure": 1012.19,
    "rainfall": 0,
    "rainrate": 0,
    "temperature": 32.37,
    "timestamp": 1780807375,
    "volt": 4.16
  }
}
